# Install Required Libraries

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.8 MB/s eta 0:00:00


# Load and Display Dataset

In [2]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


# Handle Missing Values

In [3]:
import numpy as np

cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

df.fillna(df.mean(), inplace=True)

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


# Split and Scale Data

In [4]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


# Define Objective Function for Hyperparameter Tuning with RandomForestClassifier

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score

# Run Optuna Study using TPE Sampler

In [6]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-04-16 14:23:21,171] A new study created in memory with name: no-name-5b06732f-4937-4d84-b8a0-5d265bdd7b71
[I 2026-04-16 14:23:22,282] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 167, 'max_depth': 15}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-04-16 14:23:23,992] Trial 1 finished with value: 0.756052141527002 and parameters: {'n_estimators': 183, 'max_depth': 6}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-04-16 14:23:25,288] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 119, 'max_depth': 13}. Best is trial 2 with value: 0.7709497206703911.
[I 2026-04-16 14:23:26,075] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 83, 'max_depth': 16}. Best is trial 2 with value: 0.7709497206703911.
[I 2026-04-16 14:23:27,127] Trial 4 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 185, 'max_depth': 8}. Best is trial 2 with value: 0.77094972

# Display Best Hyperparameters and Accuracy (TPE Sampler)

In [7]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279331
Best hyperparameters: {'n_estimators': 73, 'max_depth': 20}


# Evaluate Model with Best TPE Hyperparameters on Test Set

In [8]:
from sklearn.metrics import accuracy_score
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


# Redefine Objective Function for Random Sampler

# Run Optuna Study using Random Sampler

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score

# Display Best Hyperparameters and Accuracy (Random Sampler)

In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())
study.optimize(objective, n_trials=50)

[I 2026-04-16 14:23:55,496] A new study created in memory with name: no-name-e2f8272f-1cfa-4158-9bd1-f8e824c84c3c
[I 2026-04-16 14:23:56,141] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 104, 'max_depth': 20}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-04-16 14:23:56,742] Trial 1 finished with value: 0.7579143389199255 and parameters: {'n_estimators': 118, 'max_depth': 3}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-04-16 14:23:57,722] Trial 2 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 173, 'max_depth': 10}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-04-16 14:23:58,175] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 87, 'max_depth': 4}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-04-16 14:23:59,105] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 164, 'max_depth': 12}. Best is trial 4 with value: 0.7709497

# Evaluate Model with Best Random Sampler Hyperparameters on Test Set

In [11]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350094
Best hyperparameters: {'n_estimators': 70, 'max_depth': 17}


# Define Search Space for Grid Sampler

In [12]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


# Run Optuna Study using Grid Sampler

In [13]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

# Display Best Hyperparameters and Accuracy (Grid Sampler)

In [14]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-04-16 14:24:34,872] A new study created in memory with name: no-name-b0c9af09-b340-47f2-94f8-be9bf15f9198
[I 2026-04-16 14:24:35,403] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-04-16 14:24:36,231] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-04-16 14:24:36,528] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-04-16 14:24:37,129] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-04-16 14:24:37,754] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

# Evaluate Model with Best Grid Sampler Hyperparameters on Test Set

In [15]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


# Final Model Evaluation

In [16]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74
